In [7]:
import pandas as pd

df = pd.read_csv("AF_clean_sequences_cosine.csv")

def construct_reads(seq, k=3):
    n = len(seq)
    kmers = []
    for i in range(n - k + 1):
        kmers.append(seq[i:i+k])
    return kmers

def hamiltonpath(kmers):
    edges = []
    for i in kmers:
        for j in kmers:
            if i != j and i[1:] == j[:-1]:
                edges.append((i, j))
    return edges

def reconstruct_from_path(kmers):
    path = [kmers[0]]
    while len(path) < len(kmers):
        last = path[-1]
        found = False
        for kmer in kmers:
            if kmer not in path and last[1:] == kmer[:-1]:
                path.append(kmer)
                found = True
                break
        if not found:
            break
    sequence = path[0]
    for kmer in path[1:]:
        sequence += kmer[-1]
    return sequence

for idx in range(30):
    genome = str(df["Sequence"][idx]).strip().upper()
    kmers = construct_reads(genome, k=3)
    edges = hamiltonpath(kmers)
    reconstructed_seq = reconstruct_from_path(kmers)
    print("Sequence", idx+1)
    print("Original:", genome[:50])
    print("Reconstructed:", reconstructed_seq[:50])
    print()


Sequence 1
Original: MDAGSGPRRGGPGCAVLGGCHRHCAWALAFAAAPGAASRAGPPRALLVMA
Reconstructed: MDAGSGPRRGGPGCAVLGGCHRHCAWALAFAAAPGAASRAGPPRALLVMA

Sequence 2
Original: MEIVLVVFFGTEYVVRLWSAGCRSKYVGLWGRLRFARKPISIIDLIVVVA
Reconstructed: MEIVLVVFFGTEYVVRLWSAGCRSKYVGLWGRLRFARKPISIIDLIVVVA

Sequence 3
Original: MLANTVEKSEGQVDVEKWKFMMKTAQGGGHRTLLYGHAILLRHSYSGMYL
Reconstructed: MLANTVEKSEGQVDVESRSSTSRTLLYGHRTVPSGMYLCCLSTDKLAFDV

Sequence 4
Original: MLIMCTILTNCVFMAQHDPPPWTKYVEYTFTAIYTFESLVKILARGFCLH
Reconstructed: MLIMCTILTNCVFMAQHDPPPWTKYVEYTFTAIYTTEFVDLGNVSALRDP

Sequence 5
Original: MDSASSPPNAERKRPGWGLLLGARRGSAGLAKKCPFSLELAEGGPAGGTL
Reconstructed: MDSASSPPNAERKRPGWGLLLGARRGSAGLAKKCPFSLELAEGGPAGGTL

Sequence 6
Original: MLSDHSDSGEEDDEVVLQCTATIHKEQQKLCLAAEGFGNRLCFLESTSNS
Reconstructed: MLSDHSDSGEEDDEVVLQCTATIHKEQQKLCLAAEGFGNRLCFLESTSNS

Sequence 7
Original: FLIVLVCLIFSVLSTIEQYAALATGTLFWMEIVLVAFFGTEYVVRLWSAG
Reconstructed: FLIVLVCLIFSVLSTIEQYAALATGTLFWMEIVVRLWSAGCRSKYVVVAF

Sequence 8
Original: CVCVVV

In [11]:
import pandas as pd
from collections import defaultdict

df = pd.read_csv("AF_clean_sequences_cosine.csv")
df = df.dropna()

def construct_kmers(seq, k=3):
    kmers = []
    for i in range(len(seq) - k + 1):
        kmers.append(seq[i:i+k])
    return kmers

def build_debruijn(kmers):
    graph = defaultdict(list)
    for kmer in kmers:
        prefix = kmer[:-1]
        suffix = kmer[1:]
        graph[prefix].append(suffix)
    return graph

def find_start_node(graph):
    indeg = defaultdict(int)
    outdeg = defaultdict(int)
    
    for u in graph:
        outdeg[u] += len(graph[u])
        for v in graph[u]:
            indeg[v] += 1
    
    nodes = set(list(indeg.keys()) + list(outdeg.keys()))
    
    start = None
    for node in nodes:
        if outdeg[node] - indeg[node] == 1:
            return node
    
    return list(graph.keys())[0]

def eulerian_path(graph):
    g = {u: graph[u][:] for u in graph}
    start = find_start_node(g)
    stack = [start]
    path = []
    
    while stack:
        v = stack[-1]
        if v in g and len(g[v]) > 0:
            stack.append(g[v].pop())
        else:
            path.append(stack.pop())
    
    return path[::-1]

def reconstruct_from_euler(path):
    seq = path[0]
    for node in path[1:]:
        seq += node[-1]
    return seq

limit = min(30, len(df))

for idx in range(limit):
    genome = str(df["Sequence"].iloc[idx]).strip().upper()
    
    kmers = construct_kmers(genome, k=3)
    graph = build_debruijn(kmers)
    path = eulerian_path(graph)
    reconstructed_seq = reconstruct_from_euler(path)
    print(path)
    print("-------")
    print(graph)

['MD', 'DA', 'AL', 'LP', 'PT', 'TY', 'YE', 'EQ', 'QL', 'LT', 'TV', 'VP', 'PR', 'RR', 'RG', 'GP', 'PD', 'DE', 'EG', 'GS', 'SV', 'VN', 'NP', 'PE', 'EL', 'LF', 'FL', 'LP', 'PS', 'SN', 'NA', 'AL', 'LI', 'IT', 'TD', 'DM', 'ML', 'LH', 'HG', 'GG', 'GS', 'SG', 'GG', 'GV', 'VH', 'HV', 'VT', 'TQ', 'QP', 'PC', 'CG', 'GS', 'SG', 'GG', 'GP', 'PP', 'PR', 'RE', 'EG', 'GG', 'GS', 'SP', 'PP', 'PG', 'GS', 'SN', 'NT', 'TI', 'IG', 'GA', 'AR', 'RL', 'LA', 'AL', 'LL', 'LS', 'SL', 'LH', 'HQ', 'QL', 'LL', 'LT', 'TP', 'PI', 'IT', 'TH', 'HI', 'IS', 'SV', 'VS', 'SE', 'EK', 'KS', 'SK', 'KD', 'DR', 'RG', 'GS', 'SG', 'GF', 'FA', 'AE', 'ED', 'DK', 'KV', 'VT', 'TQ', 'QL', 'LD', 'DQ', 'QR', 'RL', 'LN', 'NR', 'RV', 'VE', 'ED', 'DL', 'LE', 'EG', 'GE', 'ET', 'TL', 'LL', 'LE', 'EV', 'VS', 'ST', 'TP', 'PH', 'HF', 'FM', 'MR', 'RT', 'TN', 'NS', 'SF', 'FA', 'AL', 'LK', 'KV', 'VI', 'IE', 'EQ', 'QY', 'YS', 'SQ', 'QG', 'GH', 'HL', 'LN', 'NL', 'LM', 'MV', 'VR', 'RI', 'IK', 'KE', 'EL', 'LQ', 'QR', 'RR', 'RL', 'LD', 'DQ', 'QS', 'SI